# Pipeline ETL Unificado — Catálogo Maestro de Productos

Este notebook consolida el flujo completo de **auditoría, limpieza, validación de calidad y exportación** del catálogo maestro de productos farmacéuticos:

1. **Configuración y carga** del dataset crudo (`scrap/data/catalogo_productos_raw.csv`) preservando `codigo` como `str`.
2. **Auditoría de codificación y formato**, detectando caracteres *mojibake* (CP850) y verificando integridad de parseo.
3. **Limpieza y normalización** de las 7 columnas estructurales (`codigo`, `producto`, `droga`, `tipo`, `accion`, `clasificacion`, `laboratorio`).
4. **Validación de calidad y aptitud PostgreSQL (`COPY`)**: unicidad de clave primaria, ausencia de caracteres de control y completitud.
5. **Exportación y verificación (`assert`)** del catálogo final anonimizado en `etl/data/processed/catalogo.parquet`.

In [1]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

_cwd = Path.cwd().resolve()
ROOT_DIR = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "scrap").is_dir() and (p / "etl").is_dir()),
    _cwd,
)

INPUT_PATH = ROOT_DIR / "scrap" / "data" / "catalogo_productos_raw.csv"
INTERIM_PATH = ROOT_DIR / "etl" / "data" / "interim" / "limpio.parquet"
FINAL_PATH = ROOT_DIR / "etl" / "data" / "processed" / "catalogo.parquet"

STRUCTURAL_COLUMNS = [
    "codigo",
    "producto",
    "droga",
    "tipo",
    "accion",
    "clasificacion",
    "laboratorio",
]
TEXT_COLUMNS = ["producto", "droga", "tipo", "accion", "clasificacion", "laboratorio"]

EXPECTED_ROWS = 30326
EXPECTED_COLS = 7

## 1. Carga y Auditoría de Formato (Solo Lectura)

In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo de entrada: {INPUT_PATH.relative_to(ROOT_DIR).as_posix()}")

df_raw = pd.read_csv(
    INPUT_PATH,
    encoding="utf-8",
    dtype={"codigo": str},
    usecols=STRUCTURAL_COLUMNS,
)

print(f"Dimensiones cargadas (7 columnas estructurales): {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
print(f"Nulos por columna en origen:\n{df_raw.isna().sum().to_string()}")

Dimensiones cargadas (7 columnas estructurales): 30,326 filas x 7 columnas
Nulos por columna en origen:
codigo               0
producto             0
droga            19819
tipo                 0
accion           19816
clasificacion    19816
laboratorio          0


## 2. Limpieza de Mojibake CP850 y Espacios Múltiples

In [3]:
MOJIBAKE_MAP = {
    "\xa2": "Ó",
    "\xa0": "Á",
    "\x82": "É",
    "\xa1": "Í",
    "\x81": "Ü",
    "\xa4": "Ñ",
    "\xa3": "Ú",
}
MOJIBAKE_TRANS = str.maketrans(MOJIBAKE_MAP)
MULTI_SPACE_RE = re.compile(r" {2,}")


def count_mojibake(df: pd.DataFrame) -> int:
    total = 0
    for col in TEXT_COLUMNS:
        non_null = df[col].dropna().astype(str)
        for bad_char in MOJIBAKE_MAP:
            total += int(non_null.str.contains(bad_char, regex=False).sum())
    return total


def clean_text_series(series: pd.Series) -> pd.Series:
    mask = series.notna()
    cleaned = series.copy()
    cleaned.loc[mask] = (
        series.loc[mask]
        .astype(str)
        .str.translate(MOJIBAKE_TRANS)
        .str.replace(MULTI_SPACE_RE, " ", regex=True)
        .str.strip()
    )
    return cleaned


mojibake_antes = count_mojibake(df_raw)
df_clean = df_raw.copy()
df_clean["codigo"] = df_clean["codigo"].astype(str).str.strip()
for col in TEXT_COLUMNS:
    df_clean[col] = clean_text_series(df_clean[col])

mojibake_despues = count_mojibake(df_clean)
print(f"Ocurrencias de mojibake CP850 antes   : {mojibake_antes:,}")
print(f"Ocurrencias de mojibake CP850 después : {mojibake_despues:,}")

INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(INTERIM_PATH, index=False, compression="snappy")

Ocurrencias de mojibake CP850 antes   : 12,747
Ocurrencias de mojibake CP850 después : 0


## 3. Validación de Calidad e Integridad (PostgreSQL-ready)

In [4]:
control_chars = Counter()
for col in TEXT_COLUMNS:
    for val in df_clean[col].dropna().astype(str):
        for ch in val:
            code = ord(ch)
            if code < 32 or code == 127 or ch == "\xa0":
                control_chars[repr(ch)] += 1

codigo_nulls = int(df_clean["codigo"].isna().sum())
codigo_unique = int(df_clean["codigo"].nunique(dropna=True))

print(f"Caracteres de control detectados : {dict(control_chars) if control_chars else 0}")
print(f"Nulos en 'codigo'                : {codigo_nulls}")
print(f"Valores únicos en 'codigo'       : {codigo_unique:,} / {len(df_clean):,}")
print("\nDistribución de 'tipo':")
print(df_clean["tipo"].value_counts(dropna=False).to_string())

Caracteres de control detectados : 0
Nulos en 'codigo'                : 0
Valores únicos en 'codigo'       : 30,326 / 30,326

Distribución de 'tipo':
tipo
MEDICAMENTOS             10123
REGALERIA                 9202
PERFUMERIA                8538
P.M.5                      955
P.M.                       478
ACCESORIOS                 416
ARTICULOS DE LIMPIEZA      272
ALIMENTOS                  192
VARIOS                      59
INSECTICIDAS                53
IN VITRO                    31
VARIOS SIN DESCUENTO         7


## 4. Exportación del Catálogo Final Anonimizado y Validaciones `assert`

In [5]:
FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean[STRUCTURAL_COLUMNS].to_parquet(FINAL_PATH, index=False, compression="snappy")

df_check = pd.read_parquet(FINAL_PATH)

assert df_check.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Dimensiones inesperadas: {df_check.shape}, se esperaba ({EXPECTED_ROWS}, {EXPECTED_COLS})"
)
assert list(df_check.columns) == STRUCTURAL_COLUMNS, (
    f"Orden de columnas inesperado: {list(df_check.columns)}"
)
assert count_mojibake(df_check) == 0, "Se detectó mojibake residual"
assert len(control_chars) == 0, f"Se detectaron caracteres de control: {dict(control_chars)}"
assert int(df_check["codigo"].isna().sum()) == 0, "Hay nulos en 'codigo'"
assert int(df_check["codigo"].nunique()) == EXPECTED_ROWS, "Hay códigos duplicados"
assert int(df_check["producto"].isna().sum()) == 0, "Hay nulos en 'producto'"
assert int(df_check["tipo"].isna().sum()) == 0, "Hay nulos en 'tipo'"
assert int(df_check["laboratorio"].isna().sum()) == 0, "Hay nulos en 'laboratorio'"

print(f"[OK] Dimensiones       : {df_check.shape[0]:,} filas x {df_check.shape[1]} columnas")
print(f"[OK] Columnas          : {list(df_check.columns)}")
print(f"[OK] Clave primaria    : {df_check['codigo'].nunique():,} códigos únicos, 0 nulos")
print(f"[OK] Mojibake residual : 0")

[OK] Dimensiones       : 30,326 filas x 7 columnas
[OK] Columnas          : ['codigo', 'producto', 'droga', 'tipo', 'accion', 'clasificacion', 'laboratorio']
[OK] Clave primaria    : 30,326 códigos únicos, 0 nulos
[OK] Mojibake residual : 0
